# Group-wise asymmetric int4 quantization with nibble packing

Weight-only 4-bit formats (the kind GPTQ and AWQ produce) give every **group** of `g` consecutive weights in a row its own scale and zero point, and store two 4-bit codes per byte. Decode is memory-bound, so reading a quarter of the bytes of fp16 makes it much faster even though the kernel must unpack and dequantize on the fly.

Write `quantize_int4(W, group_size)` in PyTorch.

## Inputs
- `W`: float64 `(out, in)`. `group_size` $g$ divides `in`, and both are even. Group $j$ of row $r$ is `W[r, j*g:(j+1)*g]`.

## Per group (asymmetric, codes 0..15)
1. Range that always contains 0: $\;lo = \min(\min w, 0)$, $\;hi = \max(\max w, 0)$.
2. Scale $s = (hi - lo) / 15$; if $s = 0$ (an all-zero group) use $s = 1$.
3. Zero point $z = \mathrm{clamp}(\mathrm{round}(-lo / s), 0, 15)$, the code that represents 0.
4. Codes $q = \mathrm{clamp}(\mathrm{round}(w / s) + z,\, 0,\, 15)$; dequantized $\hat w = (q - z)\,s$.

Round half to even (`torch.round`) everywhere.

## Packing
Byte $b$ of a row holds codes $2b$ and $2b + 1$: `packed[r, b] = q[r, 2b] | (q[r, 2b+1] << 4)` (even index in the **low** nibble).

## Output
A tuple `(packed, scales, zeros, W_hat, max_err)`:
- `packed`: uint8 `(out, in/2)`; `scales`: float64 `(out, in/g)`; `zeros`: uint8 `(out, in/g)`.
- `W_hat`: float64 `(out, in)`, the dequantized weights (compute it **from `packed`**, by unpacking, so a packing bug shows up here too).
- `max_err`: Python float, $\max |W - \hat W|$.

No Python loops over rows or groups.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def quantize_int4(W, group_size):
    """Return (packed uint8 (out,in/2), scales (out,in/g), zeros uint8 (out,in/g), W_hat (out,in), max_err)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[0.0, 1.5, -1.5, 0.3]], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[248, 144]], "dtype": "uint8"}, {"__nd__": [[0.2]], "dtype": "float64"}, {"__nd__": [[8]], "dtype": "uint8"}, {"__nd__": [[0.0, 1.4000000000000001, -1.6, 0.2]], "dtype": "float64"}, 0.10000000000000009]}}, {"a": [{"__nd__": [[1.0, 2.0, 3.0, 4.0], [-1.0, -2.0, 0.0, 0.0]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[248, 251], [7, 0]], "dtype": "uint8"}, {"__nd__": [[0.13333333333333333, 0.26666666666666666], [0.13333333333333333, 1.0]], "dtype": "float64"}, {"__nd__": [[0, 0], [15, 0]], "dtype": "uint8"}, {"__nd__": [[1.0666666666666667, 2.0, 2.933333333333333, 4.0], [-1.0666666666666667, -2.0, 0.0, 0.0]], "dtype": "float64"}, 0.06666666666666687]}}, {"a": [{"__nd__": [[-0.806906064357509, 1.0513613420575656, -1.0815694650959309, 1.3633941010455048, -0.17884861158809381, -0.07401799478536966, 0.3690798084594048, 0.4188652180661109, 2.0004365404271423, 0.18845722869165182, -1.559725499589318, 0.3568294190996859, -0.9714114917550105, -0.46613210607722055, -0.45383398881421744, 0.0970614513246115], [1.2228292265855303, 0.08003138453497977, -0.44283977617336534, -0.8519112459759735, -0.8099374156373064, -0.477564332678312, 0.20130307395577646, 0.09794282170827796, 0.7757895187330262, -2.159285551602563, 0.7371731772859371, 0.6576628216215121, -2.34190193455917, 1.121353007222596, -0.00879226173397203, -1.4415389552988334], [0.30152779389645123, -1.2299856011044892, -0.21809420785613604, 0.5544072684342621, 1.1002355278787779, 1.0050070612787712, -0.20611176731258388, -2.2084001858712745, -0.70596158427304, 0.26642744867095547, 0.7529094398140547, 0.11904478242458337, -1.089750378188543, -0.40565933699643625, -1.1231166392610796, -0.07144622617260332]], "dtype": "float64"}, 8], "e": {"__tup__": [{"__nd__": [[210, 240, 118, 169, 143, 144, 83, 117], [127, 3, 48, 119, 29, 221, 240, 74], [75, 217, 255, 9, 179, 175, 96, 128]], "dtype": "uint8"}, {"__nd__": [[0.1629975710760957, 0.23734413600109736], [0.13831603150410027, 0.23088366278545105], [0.2205757142500035, 0.12506840527167562]], "dtype": "float64"}, {"__nd__": [[7, 7], [6, 10], [10, 9]], "dtype": "uint8"}, {"__nd__": [[-0.8149878553804785, 0.9779854264565742, -1.14098299753267, 1.3039805686087655, -0.1629975710760957, 0.0, 0.3259951421521914, 0.4889927132282871, 1.8987530880087788, 0.23734413600109736, -1.6614089520076816, 0.4746882720021947, -0.9493765440043894, -0.4746882720021947, -0.4746882720021947, 0.0], [1.2448442835369025, 0.13831603150410027, -0.4149480945123008, -0.8298961890246016, -0.8298961890246016, -0.4149480945123008, 0.13831603150410027, 0.13831603150410027, 0.6926509883563532, -2.0779529650690596, 0.6926509883563532, 0.6926509883563532, -2.3088366278545105, 1.1544183139272552, 0.0, -1.3853019767127064], [0.2205757142500035, -1.3234542855000209, -0.2205757142500035, 0.6617271427500104, 1.1028785712500175, 1.1028785712500175, -0.2205757142500035, -2.205757142500035, -0.7504104316300537, 0.25013681054335124, 0.7504104316300537, 0.12506840527167562, -1.1256156474450805, -0.37520521581502686, -1.1256156474450805, -0.12506840527167562]], "dtype": "float64"}, 0.1178588529025088]}}, {"a": [{"__nd__": [[-0.5667637694627934, -1.056574181365729, 1.0620799269660266, -1.1293521644619289, -1.7794053080954766, 0.6549562343553932, 0.8412512124004223, -0.9491101980883807, -0.5653303274327639, 2.7797272778225293, -1.017438353321205, -0.5057891233778394, 1.1787562121345414, 0.7454060896119429, 1.6118038531320131, 0.8177558288820161, -0.4076790057527143, 0.11844004864800509, -0.5501276677481366, 0.8057569311400532, 0.5083564008622178, -1.2307981527800078, 0.48452834006261236, -0.8193381709225583, 0.07219039528984877, -0.38102026383202714, 0.28305163640424036, -2.5956068041941673, 0.4443964437476906, -1.4232256385230824, 1.172044508718855, -1.1824776995146413], [-0.5179637734213531, -0.7100846388220989, 0.25725023298212757, 1.0592385099938522, 0.21323181647187764, 9.0, 0.16465023398055598, -0.10464085423778893, 0.054739837359832455, -0.737398832629542, -0.13434217115078434, 1.4720550033090283, 1.11760343418821, -1.7048850379482186, -0.21112701426097724, 0.2052786839798478, -0.5214464942999019, -0.6513322751979604, -0.03925357409930509, 1.0155750801437982, 1.0835871287168186, 0.06190035549251886, 0.8586049345325503, -0.0781998045368921, 0.41390526978547715, -0.18988621312583207, -1.5125873725943666, -0.6628860386457914, 0.3992318151202205, 1.191400528815293, 0.356322926644702, 2.103758792262667], [0.9529293813309679, 0.07806254150545179, -0.4185079728004706, 2.3035948114893148, -0.19698457371105876, -1.1008867625482335, 0.4240606407785467, -0.5360121849411911, 0.6286530929987543, -1.5325066269200702, 0.932472317664782, 1.0127750855806499, -0.3425400485235558, -0.9572825391737342, 0.15243511378848418, 1.1884897321540928, -0.5825433640292944, -0.17130432920510302, 1.188420436176652, 0.6206063155842667, -0.1092068094373554, 0.3603389263629428, 0.19626776474722546, -0.40484523369493197, -0.1348611464621867, -2.663576702854014, 0.1302929978845863, 0.8937222195133795, 0.6832270528894249, -0.4269229140436638, 1.685053729411666, 1.3822830790890113], [-0.8226597033147672, -0.018943497032661443, -0.4349630343234197, -0.823463236396325, 1.0542624414065498, 1.0301522485546686, -0.6721085506522346, -0.14206130058129207, 0.4213707562924762, -0.2995345557279643, 0.9848403854037826, 0.35335178806542283, 0.36339809364511655, -1.513570783661183, 1.218148312791701, -2.2636934629251844, -0.06876127187819651, 1.108692872732199, 0.10089243982025005, -0.7875554808222357, -0.44443198617875523, -1.2622210707982593, 1.2413873815837857, 0.41869702041794804, -0.9245230149047061, 1.8010273500019505, -0.5432035015054584, 0.38846598281619266, -1.8446612387135426, -0.06702656838530585, 0.6382720489042668, 0.1112589188501853]], "dtype": "float64"}, 8], "e": {"__tup__": [{"__nd__": [[54, 63, 192, 77, 242, 32, 121, 122, 166, 245, 13, 61, 138, 11, 76, 95], [0, 49, 241, 17, 88, 247, 13, 151, 1, 246, 127, 93, 88, 48, 184, 247], [89, 243, 4, 55, 11, 237, 54, 249, 64, 175, 132, 39, 9, 201, 139, 239], [112, 4, 255, 98, 156, 206, 60, 15, 248, 57, 5, 191, 244, 166, 128, 139]], "dtype": "uint8"}, {"__nd__": [[0.18943234900410022, 0.2531443754095823, 0.13577033892800408, 0.2511767541942015], [0.6473389759214733, 0.2117960027504831, 0.1156612935943186, 0.24108974432380226], [0.22696543826916987, 0.18139975727161087, 0.1180642533470631, 0.2899086954843787], [0.12518171185352497, 0.23212278504779238, 0.166907230158803, 0.2430459059143662]], "dtype": "float64"}, {"__nd__": [[9, 4, 9, 10], [1, 8, 6, 6], [5, 8, 5, 9], [7, 10, 8, 8]], "dtype": "uint8"}, {"__nd__": [[-0.5682970470123007, -1.1365940940246013, 1.1365940940246013, -1.1365940940246013, -1.704891141036902, 0.5682970470123007, 0.7577293960164009, -0.9471617450205011, -0.5062887508191646, 2.7845881295054054, -1.0125775016383292, -0.5062887508191646, 1.2657218770479115, 0.7594331262287468, 1.5188662524574936, 0.7594331262287468, -0.40731101678401227, 0.13577033892800408, -0.5430813557120163, 0.8146220335680245, 0.5430813557120163, -1.2219330503520367, 0.5430813557120163, -0.8146220335680245, 0.0, -0.502353508388403, 0.2511767541942015, -2.5117675419420147, 0.502353508388403, -1.507060525165209, 1.2558837709710073, -1.2558837709710073], [-0.6473389759214733, -0.6473389759214733, 0.0, 1.2946779518429465, 0.0, 9.062745662900626, 0.0, 0.0, 0.0, -0.6353880082514494, -0.2117960027504831, 1.4825720192533818, 1.0589800137524155, -1.6943680220038648, -0.2117960027504831, 0.2117960027504831, -0.578306467971593, -0.6939677615659117, 0.0, 1.0409516423488674, 1.0409516423488674, 0.1156612935943186, 0.8096290551602302, -0.1156612935943186, 0.4821794886476045, -0.24108974432380226, -1.4465384659428135, -0.7232692329714068, 0.4821794886476045, 1.2054487216190113, 0.24108974432380226, 2.1698076989142203], [0.9078617530766795, 0.0, -0.45393087653833974, 2.269654382691699, -0.22696543826916987, -1.1348271913458494, 0.45393087653833974, -0.45393087653833974, 0.5441992718148326, -1.451198058172887, 0.9069987863580543, 1.0883985436296653, -0.36279951454322173, -0.9069987863580543, 0.18139975727161087, 1.269798300901276, -0.5903212667353155, -0.1180642533470631, 1.180642533470631, 0.5903212667353155, -0.1180642533470631, 0.35419276004118927, 0.2361285066941262, -0.35419276004118927, 0.0, -2.609178259359408, 0.0, 0.8697260864531361, 0.5798173909687574, -0.2899086954843787, 1.7394521729062722, 1.4495434774218934], [-0.8762719829746748, 0.0, -0.3755451355605749, -0.8762719829746748, 1.0014536948281998, 1.0014536948281998, -0.6259085592676249, -0.12518171185352497, 0.46424557009558476, -0.23212278504779238, 0.9284911401911695, 0.46424557009558476, 0.46424557009558476, -1.6248594953345465, 1.160613925238962, -2.321227850477924, 0.0, 1.1683506111116209, 0.166907230158803, -0.834536150794015, -0.5007216904764089, -1.335257841270424, 1.1683506111116209, 0.5007216904764089, -0.9721836236574648, 1.7013213414005635, -0.4860918118287324, 0.4860918118287324, -1.9443672473149296, 0.0, 0.7291377177430987, 0.0]], "dtype": "float64"}, 0.25725023298212757]}}, {"a": [{"__nd__": [[-0.5667637694627934, -1.056574181365729, 1.0620799269660266, -1.1293521644619289, -1.7794053080954766, 0.6549562343553932, 0.8412512124004223, -0.9491101980883807, -0.5653303274327639, 2.7797272778225293, -1.017438353321205, -0.5057891233778394, 1.1787562121345414, 0.7454060896119429, 1.6118038531320131, 0.8177558288820161, -0.4076790057527143, 0.11844004864800509, -0.5501276677481366, 0.8057569311400532, 0.5083564008622178, -1.2307981527800078, 0.48452834006261236, -0.8193381709225583, 0.07219039528984877, -0.38102026383202714, 0.28305163640424036, -2.5956068041941673, 0.4443964437476906, -1.4232256385230824, 1.172044508718855, -1.1824776995146413], [-0.5179637734213531, -0.7100846388220989, 0.25725023298212757, 1.0592385099938522, 0.21323181647187764, 9.0, 0.16465023398055598, -0.10464085423778893, 0.054739837359832455, -0.737398832629542, -0.13434217115078434, 1.4720550033090283, 1.11760343418821, -1.7048850379482186, -0.21112701426097724, 0.2052786839798478, -0.5214464942999019, -0.6513322751979604, -0.03925357409930509, 1.0155750801437982, 1.0835871287168186, 0.06190035549251886, 0.8586049345325503, -0.0781998045368921, 0.41390526978547715, -0.18988621312583207, -1.5125873725943666, -0.6628860386457914, 0.3992318151202205, 1.191400528815293, 0.356322926644702, 2.103758792262667], [0.9529293813309679, 0.07806254150545179, -0.4185079728004706, 2.3035948114893148, -0.19698457371105876, -1.1008867625482335, 0.4240606407785467, -0.5360121849411911, 0.6286530929987543, -1.5325066269200702, 0.932472317664782, 1.0127750855806499, -0.3425400485235558, -0.9572825391737342, 0.15243511378848418, 1.1884897321540928, -0.5825433640292944, -0.17130432920510302, 1.188420436176652, 0.6206063155842667, -0.1092068094373554, 0.3603389263629428, 0.19626776474722546, -0.40484523369493197, -0.1348611464621867, -2.663576702854014, 0.1302929978845863, 0.8937222195133795, 0.6832270528894249, -0.4269229140436638, 1.685053729411666, 1.3822830790890113], [-0.8226597033147672, -0.018943497032661443, -0.4349630343234197, -0.823463236396325, 1.0542624414065498, 1.0301522485546686, -0.6721085506522346, -0.14206130058129207, 0.4213707562924762, -0.2995345557279643, 0.9848403854037826, 0.35335178806542283, 0.36339809364511655, -1.513570783661183, 1.218148312791701, -2.2636934629251844, -0.06876127187819651, 1.108692872732199, 0.10089243982025005, -0.7875554808222357, -0.44443198617875523, -1.2622210707982593, 1.2413873815837857, 0.41869702041794804, -0.9245230149047061, 1.8010273500019505, -0.5432035015054584, 0.38846598281619266, -1.8446612387135426, -0.06702656838530585, 0.6382720489042668, 0.1112589188501853]], "dtype": "float64"}, 32], "e": {"__tup__": [{"__nd__": [[69, 74, 146, 73, 245, 100, 154, 155, 118, 149, 72, 88, 103, 8, 56, 74], [17, 50, 242, 34, 18, 66, 4, 34, 17, 50, 36, 35, 35, 16, 67, 82], [139, 247, 87, 105, 58, 187, 87, 200, 118, 172, 152, 121, 8, 184, 122, 205], [133, 86, 204, 118, 122, 156, 41, 12, 200, 88, 54, 173, 245, 150, 129, 138]], "dtype": "uint8"}, {"__nd__": [[0.3583556054677797], [0.7136590025298812], [0.33114476762288864], [0.27098138752847567]], "dtype": "float64"}, {"__nd__": [[7], [2], [8], [8]], "dtype": "uint8"}, {"__nd__": [[-0.7167112109355595, -1.0750668164033392, 1.0750668164033392, -1.0750668164033392, -1.7917780273388986, 0.7167112109355595, 0.7167112109355595, -1.0750668164033392, -0.7167112109355595, 2.866844843742238, -1.0750668164033392, -0.3583556054677797, 1.0750668164033392, 0.7167112109355595, 1.433422421871119, 0.7167112109355595, -0.3583556054677797, 0.0, -0.7167112109355595, 0.7167112109355595, 0.3583556054677797, -1.0750668164033392, 0.3583556054677797, -0.7167112109355595, 0.0, -0.3583556054677797, 0.3583556054677797, -2.508489238274458, 0.3583556054677797, -1.433422421871119, 1.0750668164033392, -1.0750668164033392], [-0.7136590025298812, -0.7136590025298812, 0.0, 0.7136590025298812, 0.0, 9.277567032888456, 0.0, 0.0, 0.0, -0.7136590025298812, 0.0, 1.4273180050597625, 1.4273180050597625, -1.4273180050597625, 0.0, 0.0, -0.7136590025298812, -0.7136590025298812, 0.0, 0.7136590025298812, 1.4273180050597625, 0.0, 0.7136590025298812, 0.0, 0.7136590025298812, 0.0, -1.4273180050597625, -0.7136590025298812, 0.7136590025298812, 1.4273180050597625, 0.0, 2.140977007589644], [0.9934343028686659, 0.0, -0.33114476762288864, 2.3180133733602206, -0.33114476762288864, -0.9934343028686659, 0.33114476762288864, -0.6622895352457773, 0.6622895352457773, -1.6557238381144432, 0.9934343028686659, 0.9934343028686659, -0.33114476762288864, -0.9934343028686659, 0.0, 1.3245790704915545, -0.6622895352457773, -0.33114476762288864, 1.3245790704915545, 0.6622895352457773, 0.0, 0.33114476762288864, 0.33114476762288864, -0.33114476762288864, 0.0, -2.649158140983109, 0.0, 0.9934343028686659, 0.6622895352457773, -0.33114476762288864, 1.6557238381144432, 1.3245790704915545], [-0.812944162585427, 0.0, -0.5419627750569513, -0.812944162585427, 1.0839255501139027, 1.0839255501139027, -0.5419627750569513, -0.27098138752847567, 0.5419627750569513, -0.27098138752847567, 1.0839255501139027, 0.27098138752847567, 0.27098138752847567, -1.625888325170854, 1.0839255501139027, -2.1678511002278054, 0.0, 1.0839255501139027, 0.0, -0.812944162585427, -0.5419627750569513, -1.3549069376423784, 1.3549069376423784, 0.5419627750569513, -0.812944162585427, 1.8968697126993297, -0.5419627750569513, 0.27098138752847567, -1.8968697126993297, 0.0, 0.5419627750569513, 0.0]], "dtype": "float64"}, 0.356322926644702]}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[0, 0, 0, 0], [0, 0, 0, 0]], "dtype": "uint8"}, {"__nd__": [[1.0, 1.0], [1.0, 1.0]], "dtype": "float64"}, {"__nd__": [[0, 0], [0, 0]], "dtype": "uint8"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, 0.0]}}, {"a": [{"__nd__": [[0.1, 0.1, 0.1, 0.1, -3.0, 7.5, 0.0, 2.25]], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[255, 255, 240, 116]], "dtype": "uint8"}, {"__nd__": [[0.006666666666666667, 0.7]], "dtype": "float64"}, {"__nd__": [[0, 4]], "dtype": "uint8"}, {"__nd__": [[0.1, 0.1, 0.1, 0.1, -2.8, 7.699999999999999, 0.0, 2.0999999999999996]], "dtype": "float64"}, 0.20000000000000018]}}, {"a": [{"__nd__": [[0.10841028909376382, 0.6621769265601489, 0.06929696177341593, 0.44332755621527375, 0.9521201563400041, 0.6182476178208464, 0.060712512700058285, 0.8409064230934963, 0.7833743356348939, 0.04950952967895539, 0.33231594990315705, 0.8748491653930254, 0.6096385620167922, 0.24045208286451802, 0.8957416208743856, 0.2114190727402825, 0.2589266166517976, 0.8153509830409276, 0.4055846220223892, 0.9449010546485778, 0.7750657168587115, 0.07405472151364412, 0.37577823579512626, 0.28995459262784185, 0.17518742688374334, 0.1547301182845966, 0.6062872436301338, 0.5679500965343227, 0.5999390008777803, 0.7713736153666689, 0.03753189760430509, 0.2802408072385859, 0.5639327549650055, 0.7697208665761741, 0.691684371809501, 0.8153913048019313, 0.8698319721148281, 0.44443663156181656, 0.24522258198732305, 0.5382509343636588, 0.2068376555686816, 0.7273040703281028, 0.9207458195144372, 0.3320905080785852, 0.9862184093647275, 0.07680545121126359, 0.30235927821712105, 0.05744018780520499, 0.428704222128686, 0.7972007684390989, 0.010470860982458974, 0.5232704651872665, 0.6149208785848299, 0.4837474909846424, 0.18746909142501345, 0.09117448190841648, 0.5196785852085407, 0.814998526597146, 0.5465104739475868, 0.2528147102131121, 0.5549479145286783, 0.29105283308211305, 0.460334942652155, 0.09758627647336049], [0.093289812976184, 0.6726449000239189, 0.25123810893744947, 0.40897097756656775, 0.996971849040725, 0.7420235415612734, 0.12553918262651242, 0.7833152153383834, 0.5032375967056655, 0.6594929601604786, 0.5687743949562588, 0.4730996413934023, 0.2202361384395496, 0.6786753358315863, 0.08343086112422327, 0.31540045164172925, 0.7209204723265923, 0.27077856965590164, 0.525882568487786, 0.41537464415814784, 0.8684839713664122, 0.33563778835166636, 0.6329145894532864, 0.7680591448814473, 0.4368614796486313, 0.54827823232335, 0.5153511138530742, 0.8590078977373363, 0.25913280393431126, 0.9359695325363577, 0.8196553298219781, 0.4602647016658351, 0.0821607796318563, 0.4575667296985064, 0.6834159671952454, 0.49065051132544124, 0.25304997722046196, 0.6933803125555658, 0.10283784534040386, 0.8987067338936429, 0.9669264593692809, 0.8205050664313309, 0.9262254507726398, 0.5388952909043058, 0.9945873011750779, 0.8717192411596435, 0.8384835125120396, 0.4509190761253068, 0.3369702452832899, 0.4658077396446294, 0.8234773343449406, 0.7475869976245293, 0.28424998254050426, 0.3438889121425034, 0.3528615908533881, 0.366621100636491, 0.3616069604792773, 0.06597660292105023, 0.6238356681277196, 0.8690525735304656, 0.8837372195296174, 0.8839279663438404, 0.4066085240482321, 0.6385192599463283], [0.41728544588977, 0.14335476168636352, 0.4653357040915439, 0.8123965011142429, 0.5032381794094393, 0.4439031918238888, 0.44191220042979606, 0.5887736103235008, 0.13411707854501853, 0.14928373124144712, 0.5197960964955004, 0.508926321818111, 3.725985881641858e-05, 0.5223470257043848, 0.4755905412163739, 0.7246335788867817, 0.5828767274744711, 0.8765711810574851, 0.7446926653101434, 0.20618952711118688, 0.2832084887503582, 0.6967632154150833, 0.8874987124692807, 0.5879600906260958, 0.40062087480588116, 0.8158396995274221, 0.6886677274508817, 0.7822604979721828, 0.20060561427257684, 0.04147292848591555, 0.7450643303295881, 0.23304859890074192, 0.7485069715078408, 0.013154313060112499, 0.7037967507100025, 0.6798273054056418, 0.9897204649690711, 0.41805572409834557, 0.6893429774892464, 0.12886356838399937, 0.4388710683819125, 0.8618596508083799, 0.2958533291065142, 0.36452454909482257, 0.2824618902618474, 0.07943537329893757, 0.3881490136636968, 0.6319381690071649, 0.7577578804179086, 0.19776331636921984, 0.5971664038055611, 0.8295711886674174, 0.4264521190859808, 0.12025026943301098, 0.761493105960394, 0.5250076383618008, 0.16633108640556848, 0.7582766032199514, 0.5727792022911503, 0.39712670748053114, 0.9005680573665785, 0.41423101031833076, 0.23022744812679619, 0.06875887966075878], [0.06725481408418632, 0.6405778941002952, 0.2394975559417516, 0.883070361998775, 0.14998966684054738, 0.12545782716393927, 0.6495861818228508, 0.2209816594096935, 0.025637448771886007, 0.023283025036883953, 0.9305033262872516, 0.9560124235487676, 0.042563897395490935, 0.6737505616071292, 0.5482044484345193, 0.9628686787030314, 0.11572323628230208, 0.8781468619386597, 0.9690766771920074, 0.5017796780083282, 0.014847929118043335, 0.9284100397818841, 0.29431575452678893, 0.5483083342841458, 0.2720126387988042, 0.4254624055047923, 0.7427638378128858, 0.616107267013367, 0.1663134805103118, 0.04464524457368879, 0.37686284583536156, 0.3519937913466834, 0.6878633781543976, 0.048004487226033454, 0.29999362476260916, 0.029114062792828355, 0.7339286901510401, 0.007978895924833918, 0.3137766401450156, 0.6980778906226951, 0.814277059887874, 0.3104553104135771, 0.6142211925962012, 0.95096655023555, 0.5431878941019996, 0.8321398482331939, 0.7063346551031658, 0.17090327180345488, 0.16340873497934494, 0.048360724782226194, 0.6282721342348981, 0.24572834400770227, 0.07591307801898228, 0.6875387815704507, 0.7603544755282545, 0.9025020063352661, 0.5999465532965037, 0.9747938869499112, 0.58148719671895, 0.1869859085752742, 0.8855773786307325, 0.26642188036084924, 0.8041648206400662, 0.7219251242282493], [0.48790938778404713, 0.019896960816758513, 0.22102596256267892, 0.621473920047501, 0.8277343102913596, 0.8247526438104619, 0.9910541625703142, 0.4148914379971145, 0.8289816659583716, 0.3630451275698031, 0.9475637694193936, 0.34010311064618637, 0.6176338661201349, 0.6939559000815045, 0.47701706250857157, 0.3470571433426499, 0.326745328045173, 0.039581986930097735, 0.29277922800036926, 0.8422824947106297, 0.2143327466382935, 0.9811636285377581, 0.7398482256604539, 0.9048147671735749, 0.22855237185407817, 0.38249974224728545, 0.6080995764427092, 0.43804817815231245, 0.4988725071814848, 0.4120895070001176, 0.15621372558965774, 0.6114587121535683, 0.37655968306722, 0.449711666489931, 0.5928617064223421, 0.48456918637340696, 0.5904489506202426, 0.31689636658924214, 0.7826495122064828, 0.5731921647947805, 0.06316578802000783, 0.6663278515205168, 0.36414507496680537, 0.15645904269600208, 0.3943575018818122, 0.3419546005897536, 0.04131915437612488, 0.7786341078034933, 0.851986845878983, 0.532468448870035, 0.6761643063170056, 0.5385701426395721, 0.2773781906508206, 0.6431466704134625, 0.6533362274300281, 0.44149293914435006, 0.45855969490347526, 0.3951318473598159, 0.6795281591538592, 0.8224055361246728, 0.20070927165246144, 0.9062016201693084, 0.26367187075844567, 0.47994535215371326]], "dtype": "float64"}, 16], "e": {"__tup__": [{"__nd__": [[162, 113, 175, 209, 28, 229, 74, 62, 212, 246, 28, 86, 35, 154, 202, 65, 201, 203, 125, 132, 179, 94, 31, 21, 248, 160, 155, 35, 250, 90, 90, 40], [161, 100, 191, 194, 168, 121, 163, 81, 76, 120, 94, 202, 151, 232, 244, 125, 113, 122, 164, 226, 207, 142, 223, 125, 134, 222, 101, 102, 22, 251, 255, 183], [56, 249, 137, 184, 50, 154, 160, 217, 250, 61, 197, 175, 231, 220, 19, 77, 11, 171, 111, 42, 215, 100, 20, 166, 61, 234, 39, 157, 211, 122, 127, 20], [161, 228, 34, 58, 0, 254, 161, 249, 226, 143, 224, 133, 116, 171, 19, 86, 27, 5, 12, 181, 93, 250, 217, 59, 19, 74, 177, 236, 249, 57, 78, 188], [7, 147, 205, 111, 93, 94, 185, 87, 21, 212, 243, 235, 99, 121, 104, 146, 151, 155, 107, 191, 209, 55, 120, 241, 158, 155, 181, 123, 120, 235, 243, 132]], "dtype": "uint8"}, {"__nd__": [[0.0634746770893336, 0.06299340364323852, 0.0657478939576485, 0.0543332351064764], [0.06646478993604833, 0.062397968835757184, 0.06630582007833853, 0.05892853108958936], [0.05415976674094953, 0.05916658083128538, 0.0659813643312714, 0.06003787049110523], [0.06419124524686876, 0.0646051118128005, 0.06339777001570333, 0.06498625912999408], [0.06607027750468761, 0.06541090856918387, 0.05217663414709885, 0.06041344134462056]], "dtype": "float64"}, {"__nd__": [[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], "dtype": "uint8"}, {"__nd__": [[0.1269493541786672, 0.6347467708933361, 0.0634746770893336, 0.4443227396253352, 0.952120156340004, 0.6347467708933361, 0.0634746770893336, 0.8251708021613369, 0.7616961250720032, 0.0634746770893336, 0.31737338544666804, 0.8886454792506704, 0.6347467708933361, 0.2538987083573344, 0.8886454792506704, 0.1904240312680008, 0.2519736145729541, 0.8189142473621007, 0.37796042185943113, 0.9449010546485779, 0.7559208437188623, 0.06299340364323852, 0.37796042185943113, 0.3149670182161926, 0.18898021092971556, 0.12598680728647704, 0.6299340364323852, 0.5669406327891466, 0.6299340364323852, 0.7559208437188623, 0.06299340364323852, 0.2519736145729541, 0.5917310456188365, 0.788974727491782, 0.7232268335341335, 0.788974727491782, 0.8547226214494306, 0.4602352577035395, 0.262991575830594, 0.525983151661188, 0.1972436818729455, 0.7232268335341335, 0.920470515407079, 0.32873946978824253, 0.9862184093647276, 0.0657478939576485, 0.32873946978824253, 0.0657478939576485, 0.4346658808518112, 0.814998526597146, 0.0, 0.543332351064764, 0.5976655861712403, 0.4889991159582876, 0.1629997053194292, 0.1086664702129528, 0.543332351064764, 0.814998526597146, 0.543332351064764, 0.271666175532382, 0.543332351064764, 0.271666175532382, 0.4346658808518112, 0.1086664702129528], [0.06646478993604833, 0.6646478993604833, 0.26585915974419333, 0.39878873961629, 0.996971849040725, 0.7311126892965316, 0.13292957987209666, 0.79757747923258, 0.5317183194883867, 0.6646478993604833, 0.598183109424435, 0.46525352955233834, 0.199394369808145, 0.6646478993604833, 0.06646478993604833, 0.33232394968024165, 0.7487756260290862, 0.24959187534302874, 0.49918375068605747, 0.43678578185030026, 0.8735715637006005, 0.31198984417878595, 0.6239796883575719, 0.7487756260290862, 0.43678578185030026, 0.5615817195218147, 0.49918375068605747, 0.8735715637006005, 0.24959187534302874, 0.9359695325363577, 0.8111735948648434, 0.43678578185030026, 0.06630582007833853, 0.46414074054836973, 0.6630582007833853, 0.46414074054836973, 0.2652232803133541, 0.6630582007833853, 0.13261164015667706, 0.9282814810967395, 0.994587301175078, 0.7956698409400624, 0.9282814810967395, 0.5304465606267083, 0.994587301175078, 0.861975661018401, 0.861975661018401, 0.46414074054836973, 0.35357118653753616, 0.4714282487167149, 0.8249994352542511, 0.7660709041646617, 0.2946426554479468, 0.35357118653753616, 0.35357118653753616, 0.35357118653753616, 0.35357118653753616, 0.05892853108958936, 0.648213841985483, 0.8839279663438404, 0.8839279663438404, 0.8839279663438404, 0.41249971762712556, 0.648213841985483], [0.43327813392759623, 0.1624793002228486, 0.48743790066854575, 0.8123965011142429, 0.48743790066854575, 0.43327813392759623, 0.43327813392759623, 0.5957574341504448, 0.10831953348189906, 0.1624793002228486, 0.5415976674094953, 0.48743790066854575, 0.0, 0.5415976674094953, 0.48743790066854575, 0.7040769676323438, 0.5916658083128538, 0.8874987124692807, 0.7691655508067099, 0.17749974249385614, 0.2958329041564269, 0.7099989699754246, 0.8874987124692807, 0.5916658083128538, 0.4141660658189976, 0.8283321316379952, 0.7099989699754246, 0.7691655508067099, 0.17749974249385614, 0.05916658083128538, 0.7691655508067099, 0.2366663233251415, 0.7257950076439854, 0.0, 0.7257950076439854, 0.659813643312714, 0.989720464969071, 0.3958881859876284, 0.659813643312714, 0.1319627286625428, 0.4618695503188998, 0.8577577363065282, 0.2639254573250856, 0.3958881859876284, 0.2639254573250856, 0.0659813643312714, 0.3958881859876284, 0.659813643312714, 0.780492316384368, 0.1801136114733157, 0.6003787049110523, 0.8405301868754733, 0.42026509343773666, 0.12007574098221047, 0.780492316384368, 0.5403408344199471, 0.1801136114733157, 0.780492316384368, 0.6003787049110523, 0.42026509343773666, 0.9005680573665785, 0.42026509343773666, 0.24015148196442093, 0.06003787049110523], [0.06419124524686876, 0.6419124524686876, 0.256764980987475, 0.8986774334561626, 0.1283824904937375, 0.1283824904937375, 0.6419124524686876, 0.19257373574060627, 0.0, 0.0, 0.8986774334561626, 0.9628686787030314, 0.06419124524686876, 0.6419124524686876, 0.5777212072218187, 0.9628686787030314, 0.129210223625601, 0.904471565379207, 0.9690766771920075, 0.516840894502404, 0.0, 0.904471565379207, 0.3230255590640025, 0.516840894502404, 0.258420447251202, 0.4522357826896035, 0.7106562299408055, 0.646051118128005, 0.19381533543840151, 0.0646051118128005, 0.38763067087680303, 0.3230255590640025, 0.6973754701727367, 0.06339777001570333, 0.3169888500785167, 0.0, 0.76077324018844, 0.0, 0.3169888500785167, 0.6973754701727367, 0.8241710102041433, 0.3169888500785167, 0.6339777001570334, 0.95096655023555, 0.57057993014133, 0.8241710102041433, 0.6973754701727367, 0.19019331004711, 0.19495877738998224, 0.06498625912999408, 0.6498625912999407, 0.2599450365199763, 0.06498625912999408, 0.7148488504299348, 0.779835109559929, 0.9098076278199171, 0.5848763321699467, 0.9747938869499111, 0.5848763321699467, 0.19495877738998224, 0.9098076278199171, 0.2599450365199763, 0.779835109559929, 0.7148488504299348], [0.4624919425328132, 0.0, 0.19821083251406282, 0.5946324975421885, 0.8589136075609389, 0.7928433300562513, 0.9910541625703141, 0.39642166502812565, 0.8589136075609389, 0.33035138752343807, 0.9249838850656265, 0.33035138752343807, 0.5946324975421885, 0.7267730525515637, 0.4624919425328132, 0.33035138752343807, 0.32705454284591934, 0.06541090856918387, 0.2616436342767355, 0.8503418113993904, 0.19623272570755163, 0.9811636285377581, 0.7195199942610225, 0.9157527199685742, 0.19623272570755163, 0.39246545141510325, 0.5886981771226548, 0.4578763599842871, 0.523287268553471, 0.39246545141510325, 0.13082181713836774, 0.5886981771226548, 0.365236439029692, 0.46958970732388966, 0.5739429756180874, 0.46958970732388966, 0.5739429756180874, 0.3130598048825931, 0.7826495122064828, 0.5739429756180874, 0.05217663414709885, 0.678296243912285, 0.365236439029692, 0.15652990244129655, 0.4174130731767908, 0.365236439029692, 0.05217663414709885, 0.7826495122064828, 0.8457881788246878, 0.543720972101585, 0.6645478547908261, 0.543720972101585, 0.3020672067231028, 0.6645478547908261, 0.6645478547908261, 0.4228940894123439, 0.4833075307569645, 0.4228940894123439, 0.6645478547908261, 0.8457881788246878, 0.18124032403386167, 0.9062016201693084, 0.24165376537848224, 0.4833075307569645]], "dtype": "float64"}, 0.03281715247005912]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(quantize_int4, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")